# 1. Mount Google Drive
Mount your drive so Colab can access the `balanced_dataset.zip` file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 2. Extract the Dataset
Make sure you have uploaded `balanced_dataset.zip` to your Google Drive.
Adjust `ZIP_PATH` if you put it inside a specific folder.

In [ ]:
import os
import zipfile

ZIP_PATH = '/content/drive/MyDrive/balanced_dataset.zip'
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction complete!")
else:
    print("Dataset already extracted.")

# 3. Data Preprocessing
Extract Mel-Spectrogram features from audio files. Classes will be mapped 0-7 as expected by your FastAPI server.

In [ ]:
import librosa
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

class_map = {
    ('id_fan_00', 'normal'): 0,
    ('id_fan_00', 'abnormal'): 1,
    ('id_pump_00', 'normal'): 2,
    ('id_pump_00', 'abnormal'): 3,
    ('id_slider_00', 'normal'): 4,
    ('id_slider_00', 'abnormal'): 5,
    ('id_valve_00', 'normal'): 6,
    ('id_valve_00', 'abnormal'): 7
}

def extract_mel_spectrogram(file_path):
    try:
        y, sr = librosa.load(file_path, sr=16000, duration=2.0)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        IMG_WIDTH = 128
        if mel_spec_db.shape[1] < IMG_WIDTH:
            pad_width = IMG_WIDTH - mel_spec_db.shape[1]
            mel_spec_db = np.pad(mel_spec_db, pad_width=((0,0), (0, pad_width)), mode='constant')
        else:
            mel_spec_db = mel_spec_db[:, :IMG_WIDTH]
        return mel_spec_db
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

X = []
y = []

print("Extracting features... this might take a while.")
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.endswith('.wav'):
            file_path = os.path.join(root, file)
            parts = file_path.replace("\\", "/").split('/')
            state = parts[-2]
            machine = parts[-3]
            
            if (machine, state) in class_map:
                label = class_map[(machine, state)]
                features = extract_mel_spectrogram(file_path)
                if features is not None:
                    X.append(features)
                    y.append(label)

X = np.array(X)
y = np.array(y)

# Reshape for CNN
X = X.reshape(-1, 128, 128, 1)
y = to_categorical(y, num_classes=8)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# 4. Build and Train Model
Training a 2D CNN model for Mel-Spectrogram features.

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 1)),
    tf.keras.layers.MaxPooling2D(2,2),
    
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(8, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_test, y_test))

# 5. Evaluate and Save
Save the H5 format and also convert to ONNX format.

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")

H5_MODEL_PATH = '/content/drive/MyDrive/audio_classification_model.h5'
model.save(H5_MODEL_PATH)
print(f"Model saved to {H5_MODEL_PATH}")

In [ ]:
!pip install -q tf2onnx
import tf2onnx
import onnx

spec = (tf.TensorSpec((None, 128, 128, 1), tf.float32, name="input"),)
output_path = "/content/drive/MyDrive/audio_classification_model.onnx"
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13, output_path=output_path)
print(f"ONNX Model saved to {output_path}")